# Advanced Problems with Solutions: Copying Python Sets

This notebook develops a deeper understanding of **shallow copying**, **deep copying**, **object identity**, **hashability**, **aliasing**, and the special issues that arise when Python sets contain user-defined objects.

## Learning goals

By the end of this notebook, you should be able to:

- distinguish assignment, shallow copying, and deep copying;
- use `set.copy()`, `set(...)`, and `{*s}` correctly;
- explain why shallow copies share contained objects;
- explain what `copy.deepcopy()` does and does not guarantee;
- reason about object identity using `is` and `id`;
- understand why set elements must be hashable;
- identify dangerous classes whose hash depends on mutable state;
- design safer hashable objects;
- reason about nested shared references and `deepcopy` memoization;
- write tests that verify copy behavior instead of relying on printed output.

> Best practice: avoid relying on set display order. Sets are unordered collections, so examples below compare values, identities, and invariants rather than exact printed ordering.


## 1. Baseline: assignment is not copying

Before discussing shallow and deep copies, remember that simple assignment creates another reference to the **same set object**.


In [1]:
original = {1, 2, 3}
alias = original

print(original is alias)
print(id(original), id(alias))

alias.add(4)

assert original == {1, 2, 3, 4}
assert alias == {1, 2, 3, 4}
assert original is alias


True
2302285314784 2302285314784


### Key idea

`alias = original` does not create a new set. Both names refer to the same object.


## 2. Three standard shallow-copy techniques

For sets, these all create a new outer set:

- `s.copy()`
- `set(s)`
- `{*s}`

The copied set is new, but the contained objects are still shared.


In [2]:
class Person:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Person(name={self.name!r})"


p1 = Person("John")
p2 = Person("Eric")
s1 = {p1, p2}

s_copy = s1.copy()
s_constructor = set(s1)
s_unpack = {*s1}

assert s_copy is not s1
assert s_constructor is not s1
assert s_unpack is not s1

# All four sets contain references to the same Person objects.
assert p1 in s_copy
assert p1 in s_constructor
assert p1 in s_unpack


In [3]:
p1.name = "John Cleese"

for label, s in [
    ("original", s1),
    ("copy()", s_copy),
    ("set()", s_constructor),
    ("unpacking", s_unpack),
]:
    print(label, s)

# Every shallow copy sees the changed Person object.
assert any(person.name == "John Cleese" for person in s_copy)
assert any(person.name == "John Cleese" for person in s_constructor)
assert any(person.name == "John Cleese" for person in s_unpack)


original {Person(name='Eric'), Person(name='John Cleese')}
copy() {Person(name='Eric'), Person(name='John Cleese')}
set() {Person(name='Eric'), Person(name='John Cleese')}
unpacking {Person(name='Eric'), Person(name='John Cleese')}


## 3. Identity-level verification

A robust way to demonstrate shallow copying is to compare element identities directly.


In [4]:
def ids_of(iterable):
    return {id(item) for item in iterable}


assert ids_of(s1) == ids_of(s_copy)
assert ids_of(s1) == ids_of(s_constructor)
assert ids_of(s1) == ids_of(s_unpack)

print("Outer set identities:")
print(id(s1), id(s_copy), id(s_constructor), id(s_unpack))

print("\nElement identities:")
print(ids_of(s1))


Outer set identities:
2302285314560 2302285315008 2302285315680 2302285312992

Element identities:
{2302283593872, 2302285647552}


## Problem 1 — Predict shallow-copy behavior

Consider the following code.

```python
class Box:
    def __init__(self, value):
        self.value = value

a = Box(10)
b = Box(20)

s1 = {a, b}
s2 = s1.copy()

a.value = 999
s2.remove(b)
```

Without running it, answer:

1. Does `s1 is s2` evaluate to `True` or `False`?
2. Does the change `a.value = 999` appear when inspecting `s1`?
3. Does the change appear when inspecting `s2`?
4. Is `b` still in `s1`?
5. Is `b` still in `s2`?


### Solution 1

1. `False` — `copy()` creates a new outer set.
2. Yes — `s1` still contains the same `a` object.
3. Yes — `s2` also contains that same `a` object.
4. Yes — removing `b` from `s2` changes only the copied set.
5. No.

A shallow copy separates **container membership**, but it does not duplicate the member objects.


In [5]:
class Box:
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f"Box({self.value})"


a = Box(10)
b = Box(20)

s1 = {a, b}
s2 = s1.copy()

a.value = 999
s2.remove(b)

assert s1 is not s2
assert a in s1 and a in s2
assert a.value == 999
assert b in s1
assert b not in s2

print("s1:", s1)
print("s2:", s2)


s1: {Box(20), Box(999)}
s2: {Box(999)}


## 4. Deep copying sets

`copy.deepcopy()` recursively copies the set and, when possible, copies its contained objects too.


In [6]:
from copy import deepcopy

p1 = Person("Michael")
p2 = Person("Eric")

original = {p1, p2}
deep = deepcopy(original)

assert deep is not original
assert ids_of(deep).isdisjoint(ids_of(original))

print("original:", original)
print("deep:", deep)


original: {Person(name='Eric'), Person(name='Michael')}
deep: {Person(name='Michael'), Person(name='Eric')}


In [7]:
p1.name = "Terry"

print("original:", original)
print("deep:", deep)

assert any(p.name == "Terry" for p in original)
assert all(p.name != "Terry" for p in deep)


original: {Person(name='Eric'), Person(name='Terry')}
deep: {Person(name='Michael'), Person(name='Eric')}


### Important nuance

A deep copy is **not** the same as “serialize everything into unrelated values.”

`deepcopy()`:

- recursively copies many mutable objects;
- preserves some immutable objects;
- preserves internal aliasing relationships using a memo table;
- may invoke custom copy behavior implemented by a class;
- can fail or behave specially for resources such as files, sockets, modules, and certain extension objects.


## Problem 2 — Prove deep independence

Create a set containing three mutable, hashable `Person` objects.

1. Make a deep copy.
2. Verify that the outer sets differ by identity.
3. Verify that no element identity is shared.
4. Mutate every original `Person`.
5. Prove that the deep copy did not change.

Do not depend on iteration order.


### Solution 2


In [8]:
people = {
    Person("Alice"),
    Person("Bob"),
    Person("Charlie"),
}

people_deep = deepcopy(people)

assert people_deep is not people
assert ids_of(people).isdisjoint(ids_of(people_deep))

for person in people:
    person.name = person.name.upper()

original_names = {person.name for person in people}
deep_names = {person.name for person in people_deep}

assert original_names == {"ALICE", "BOB", "CHARLIE"}
assert deep_names == {"Alice", "Bob", "Charlie"}

print("Original names:", original_names)
print("Deep-copy names:", deep_names)


Original names: {'BOB', 'ALICE', 'CHARLIE'}
Deep-copy names: {'Charlie', 'Alice', 'Bob'}


## 5. Why can a mutable object be inside a set?

Set elements must be **hashable**, not necessarily deeply immutable.

A normal user-defined class is hashable by object identity unless it overrides equality/hash behavior. Therefore an instance may have mutable attributes and still be legal inside a set.

That is why the simple `Person` class above works.


In [9]:
p = Person("Ada")

print("hash(p):", hash(p))

s = {p}
p.name = "Ada Lovelace"

# Membership remains valid because the default hash is based on identity.
assert p in s
print(s)


hash(p): 143892873445
{Person(name='Ada Lovelace')}


## 6. Critical pitfall: mutable state used in `__hash__`

A set assumes an element's hash stays stable while that element is stored in the set.

If you define `__hash__` from mutable attributes and later modify those attributes, the object can become effectively “lost” inside the set.


In [10]:
class DangerousPerson:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, DangerousPerson):
            return NotImplemented
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)

    def __repr__(self):
        return f"DangerousPerson({self.name!r})"


person = DangerousPerson("Alice")
people = {person}

assert person in people

old_hash = hash(person)
person.name = "Bob"
new_hash = hash(person)

print("old hash:", old_hash)
print("new hash:", new_hash)
print("person in people:", person in people)
print("set contents:", people)


old hash: -2154317884356834628
new hash: 819779227681376695
person in people: False
set contents: {DangerousPerson('Bob')}


### Why this is dangerous

The set originally placed the object according to the hash of `"Alice"`. After mutation, lookup uses the hash of `"Bob"`.

The object may still physically exist inside the set, but membership tests and removal can fail because the set's internal hash-table assumptions were violated.

**Best practice:** never let the hash of an object change while that object is used as a set element or dictionary key.


## Problem 3 — Diagnose a broken set element

A developer writes:

```python
class User:
    def __init__(self, email):
        self.email = email

    def __eq__(self, other):
        return isinstance(other, User) and self.email == other.email

    def __hash__(self):
        return hash(self.email)
```

They insert a `User` into a set, then change `user.email`.

Explain:

1. why this is unsafe;
2. what kinds of failures may occur;
3. two better designs.


### Solution 3

The object's hash depends on `email`, so changing `email` changes the hash after insertion. A set expects that hash to remain stable.

Possible consequences include:

- `user in users` unexpectedly becoming `False`;
- `users.remove(user)` raising `KeyError`;
- logically duplicate objects coexisting in confusing ways;
- generally inconsistent set behavior.

Two better designs:

1. Make the value object immutable, for example with a frozen dataclass.
2. Use identity-based hashing/equality if identity, not email value, defines object uniqueness.

If email needs to change and value-based equality is required, remove the object from the set **before** changing the field, then reinsert it afterward—but immutable value objects are usually easier to reason about.


## 7. Safer design: immutable set elements

Frozen dataclasses are excellent for value-like objects used as set elements.


In [11]:
from dataclasses import dataclass

@dataclass(frozen=True)
class UserKey:
    email: str
    tenant_id: int


u1 = UserKey("alice@example.com", 1)
u2 = UserKey("bob@example.com", 1)

users = {u1, u2}

assert u1 in users
print(users)


{UserKey(email='alice@example.com', tenant_id=1), UserKey(email='bob@example.com', tenant_id=1)}


Because the dataclass is frozen, fields cannot normally be reassigned, which protects the hash invariant.


## Problem 4 — Compare four “copies”

Given:


In [12]:
x = Person("X")
y = Person("Y")
base = {x, y}

copies = {
    "assignment": base,
    "copy_method": base.copy(),
    "constructor": set(base),
    "unpacking": {*base},
    "deepcopy": deepcopy(base),
}


Classify each result according to:

- same outer set?
- same contained objects?
- isolated from element mutation?


### Solution 4


In [13]:
base_ids = ids_of(base)

report = {}

for name, candidate in copies.items():
    report[name] = {
        "same_outer_set": candidate is base,
        "same_element_identities": ids_of(candidate) == base_ids,
        "shares_any_element_identity": bool(ids_of(candidate) & base_ids),
    }

report


{'assignment': {'same_outer_set': True,
  'same_element_identities': True,
  'shares_any_element_identity': True},
 'copy_method': {'same_outer_set': False,
  'same_element_identities': True,
  'shares_any_element_identity': True},
 'constructor': {'same_outer_set': False,
  'same_element_identities': True,
  'shares_any_element_identity': True},
 'unpacking': {'same_outer_set': False,
  'same_element_identities': True,
  'shares_any_element_identity': True},
 'deepcopy': {'same_outer_set': False,
  'same_element_identities': False,
  'shares_any_element_identity': False}}

Expected conceptual result:

| Technique | Same outer set? | Shares original elements? | Element mutations isolated? |
|---|---:|---:|---:|
| assignment | yes | yes | no |
| `copy()` | no | yes | no |
| `set(base)` | no | yes | no |
| `{*base}` | no | yes | no |
| `deepcopy(base)` | no | normally no | normally yes |


## 8. Shallow copy is often exactly what you want

Deep copying is not automatically better.

If a set contains immutable values, a shallow copy is usually enough.


In [14]:
numbers = {1, 2, 3}
copy_numbers = numbers.copy()

copy_numbers.add(4)

assert numbers == {1, 2, 3}
assert copy_numbers == {1, 2, 3, 4}

# Integers are immutable, so sharing them is harmless.
assert numbers is not copy_numbers


In [15]:
tags = {"python", "sets", "copying"}
tags2 = set(tags)

tags2.add("deepcopy")

assert "deepcopy" not in tags
assert "deepcopy" in tags2


## Problem 5 — Choose shallow or deep copy

For each case, choose the simplest appropriate technique and explain why.

1. A set of integers.
2. A set of immutable `(country_code, tax_rate)` tuples.
3. A set of mutable objects that will be edited independently.
4. A set of shared domain entities that should remain shared, but membership should differ.


### Solution 5

1. **Shallow copy** — integers are immutable.
2. **Shallow copy** — assuming all tuple contents are immutable.
3. **Deep copy**, or preferably explicit cloning tailored to the domain.
4. **Shallow copy** — you want a new set container but intentionally shared element objects.

A key engineering principle is to copy **only as deeply as the ownership model requires**.


## 9. Nested mutable state inside hashable objects

An object can use identity hashing while holding nested mutable state. Shallow copying a set still shares that entire object graph.


In [16]:
class Profile:
    def __init__(self, name, skills):
        self.name = name
        self.skills = list(skills)

    def __repr__(self):
        return f"Profile(name={self.name!r}, skills={self.skills!r})"


profile = Profile("Ada", ["Python", "Math"])
original = {profile}

shallow = original.copy()
deep = deepcopy(original)

profile.skills.append("Algorithms")

original_profile = next(iter(original))
shallow_profile = next(iter(shallow))
deep_profile = next(iter(deep))

assert original_profile.skills == ["Python", "Math", "Algorithms"]
assert shallow_profile.skills == ["Python", "Math", "Algorithms"]
assert deep_profile.skills == ["Python", "Math"]

print("original:", original)
print("shallow:", shallow)
print("deep:", deep)


original: {Profile(name='Ada', skills=['Python', 'Math', 'Algorithms'])}
shallow: {Profile(name='Ada', skills=['Python', 'Math', 'Algorithms'])}
deep: {Profile(name='Ada', skills=['Python', 'Math'])}


## Problem 6 — Two levels of aliasing

A `Project` object has a mutable list of labels. The object is stored in a set.

Build:

- `original`
- `shallow = original.copy()`
- `deep = deepcopy(original)`

Then mutate the labels list in the original project and verify exactly which collections observe the mutation.


### Solution 6


In [17]:
class Project:
    def __init__(self, name, labels):
        self.name = name
        self.labels = list(labels)

    def __repr__(self):
        return f"Project({self.name!r}, labels={self.labels!r})"


project = Project("Compiler", ["python", "advanced"])

original = {project}
shallow = original.copy()
deep = deepcopy(original)

project.labels.append("copy-semantics")

assert next(iter(original)).labels == ["python", "advanced", "copy-semantics"]
assert next(iter(shallow)).labels == ["python", "advanced", "copy-semantics"]
assert next(iter(deep)).labels == ["python", "advanced"]

print("original:", original)
print("shallow:", shallow)
print("deep:", deep)


original: {Project('Compiler', labels=['python', 'advanced', 'copy-semantics'])}
shallow: {Project('Compiler', labels=['python', 'advanced', 'copy-semantics'])}
deep: {Project('Compiler', labels=['python', 'advanced'])}


## 10. Deep copy preserves internal sharing

Suppose two objects refer to the same nested object. A correct deep copy usually creates one copied nested object and makes both copied parents refer to that same copied nested object.

This is managed by `deepcopy`'s internal memo dictionary.


In [18]:
class TeamMember:
    def __init__(self, name, shared_settings):
        self.name = name
        self.shared_settings = shared_settings

    def __repr__(self):
        return f"TeamMember({self.name!r})"


settings = {"timezone": "UTC", "language": "en"}

a = TeamMember("A", settings)
b = TeamMember("B", settings)

team = {a, b}
team_copy = deepcopy(team)

copy_a, copy_b = sorted(team_copy, key=lambda member: member.name)

assert copy_a.shared_settings is copy_b.shared_settings
assert copy_a.shared_settings is not settings

print(copy_a.shared_settings)
print(copy_b.shared_settings)


{'timezone': 'UTC', 'language': 'en'}
{'timezone': 'UTC', 'language': 'en'}


### Why preserving aliasing matters

If `deepcopy()` independently duplicated every path without memoization, shared references could accidentally become separate objects and the copied object graph would have different semantics.


## Problem 7 — Shared nested object graph

Create three objects that all reference the same mutable configuration dictionary. Put them in a set and deep-copy it.

Verify that:

1. none of the copied top-level objects are originals;
2. the copied objects share one copied configuration;
3. the copied configuration is not the original configuration;
4. changing the original configuration does not affect the deep copy.


### Solution 7


In [19]:
shared_config = {"retries": 3, "timeout": 5}

members = {
    TeamMember("Alice", shared_config),
    TeamMember("Bob", shared_config),
    TeamMember("Charlie", shared_config),
}

members_copy = deepcopy(members)

assert ids_of(members).isdisjoint(ids_of(members_copy))

copied_configs = {
    id(member.shared_settings)
    for member in members_copy
}

assert len(copied_configs) == 1
assert next(iter(members_copy)).shared_settings is not shared_config

shared_config["timeout"] = 99

assert {m.shared_settings["timeout"] for m in members} == {99}
assert {m.shared_settings["timeout"] for m in members_copy} == {5}

print("Original config:", shared_config)
print("Copied config:", next(iter(members_copy)).shared_settings)


Original config: {'retries': 3, 'timeout': 99}
Copied config: {'retries': 3, 'timeout': 5}


## 11. Cyclic references and `deepcopy`

Python's `deepcopy()` can handle many cyclic reference graphs because it memoizes objects already copied.


In [20]:
class Node:
    def __init__(self, name):
        self.name = name
        self.link = None

    def __repr__(self):
        return f"Node({self.name!r})"


node = Node("loop")
node.link = node

nodes = {node}
nodes_copy = deepcopy(nodes)

copied_node = next(iter(nodes_copy))

assert copied_node is not node
assert copied_node.link is copied_node

print(copied_node)
print(copied_node.link is copied_node)


Node('loop')
True


## Problem 8 — Deep-copy a two-node cycle

Construct two hashable-by-identity nodes:

- `a.next = b`
- `b.next = a`

Put both in a set, deep-copy the set, and verify the cycle is preserved without referencing either original node.


### Solution 8


In [21]:
a = Node("A")
b = Node("B")

a.link = b
b.link = a

cycle = {a, b}
cycle_copy = deepcopy(cycle)

copied_by_name = {node.name: node for node in cycle_copy}
ca = copied_by_name["A"]
cb = copied_by_name["B"]

assert ca is not a
assert cb is not b
assert ca.link is cb
assert cb.link is ca

print(ca, "->", ca.link)
print(cb, "->", cb.link)


Node('A') -> Node('B')
Node('B') -> Node('A')


## 12. `frozenset` and copying

A `frozenset` is immutable. Because it cannot be modified, making a shallow copy is often unnecessary.


In [22]:
f = frozenset({1, 2, 3})

f_via_constructor = frozenset(f)
f_via_copy = f.copy()

print("constructor reused object:", f_via_constructor is f)
print("copy reused object:", f_via_copy is f)

assert f_via_constructor == f
assert f_via_copy == f


constructor reused object: True
copy reused object: True


Implementation details such as whether an immutable object is reused should not usually be the basis of program logic. Prefer equality/value semantics unless identity is specifically relevant.


## Problem 9 — Set of frozensets

Why is this valid?

```python
matrix_regions = {
    frozenset({(0, 0), (0, 1)}),
    frozenset({(1, 0), (1, 1)}),
}
```

Why would replacing the inner `frozenset` objects with normal `set` objects fail?


### Solution 9

A `frozenset` is hashable when its elements are hashable, so it can itself be stored inside another set.

A normal `set` is mutable and unhashable, so a `set` cannot directly contain another `set`.

Nested set-like structures therefore commonly use `frozenset` for the inner level.


In [23]:
matrix_regions = {
    frozenset({(0, 0), (0, 1)}),
    frozenset({(1, 0), (1, 1)}),
}

assert len(matrix_regions) == 2

try:
    invalid = {{1, 2}, {3, 4}}
except TypeError as exc:
    print(type(exc).__name__, exc)


TypeError unhashable type: 'set'


## 13. Copying versus rebuilding

Sometimes the clearest solution is not `deepcopy()` but an explicit transformation.


In [24]:
@dataclass(frozen=True)
class Coordinate:
    x: int
    y: int


points = {Coordinate(1, 2), Coordinate(3, 4)}

# Explicit rebuild.
translated = {
    Coordinate(point.x + 10, point.y + 10)
    for point in points
}

assert translated == {
    Coordinate(11, 12),
    Coordinate(13, 14),
}


Explicit reconstruction is often preferable when you want **new semantic values**, not merely copied object identity.


## Problem 10 — Clone only what should be independent

Suppose each `Employee` contains:

- a mutable `preferences` dictionary that should become independent;
- an immutable `department` string that may be shared;
- a reference to one shared `Company` object that should remain shared.

A blind `deepcopy()` would also duplicate the company unless customized.

Design a method that copies an employee according to the desired ownership semantics.


### Solution 10

Use explicit cloning instead of blindly deep-copying the entire domain graph.


In [25]:
class Company:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Company({self.name!r})"


class Employee:
    def __init__(self, name, department, company, preferences):
        self.name = name
        self.department = department
        self.company = company
        self.preferences = dict(preferences)

    def clone(self):
        return Employee(
            name=self.name,
            department=self.department,
            company=self.company,                 # intentionally shared
            preferences=deepcopy(self.preferences),  # intentionally independent
        )

    def __repr__(self):
        return (
            f"Employee({self.name!r}, {self.department!r}, "
            f"{self.company!r}, {self.preferences!r})"
        )


company = Company("Example Corp")
employee = Employee(
    "Alice",
    "Engineering",
    company,
    {"editor": "vim", "alerts": ["email"]},
)

clone = employee.clone()

assert clone is not employee
assert clone.company is employee.company
assert clone.preferences is not employee.preferences
assert clone.preferences["alerts"] is not employee.preferences["alerts"]

employee.preferences["alerts"].append("sms")

assert employee.preferences["alerts"] == ["email", "sms"]
assert clone.preferences["alerts"] == ["email"]

print(employee)
print(clone)


Employee('Alice', 'Engineering', Company('Example Corp'), {'editor': 'vim', 'alerts': ['email', 'sms']})
Employee('Alice', 'Engineering', Company('Example Corp'), {'editor': 'vim', 'alerts': ['email']})


## 14. Customizing `deepcopy`

Classes can define `__deepcopy__` when generic recursive copying is not appropriate.

The `memo` argument must be respected so cyclic graphs and shared references remain correct.


In [26]:
class SharedCompanyEmployee:
    def __init__(self, name, company, preferences):
        self.name = name
        self.company = company
        self.preferences = preferences

    def __deepcopy__(self, memo):
        existing = memo.get(id(self))
        if existing is not None:
            return existing

        clone = type(self).__new__(type(self))
        memo[id(self)] = clone

        clone.name = self.name
        clone.company = self.company  # deliberately shared
        clone.preferences = deepcopy(self.preferences, memo)
        return clone

    def __repr__(self):
        return (
            f"SharedCompanyEmployee({self.name!r}, "
            f"{self.company!r}, {self.preferences!r})"
        )


company = Company("Shared Corp")
e = SharedCompanyEmployee(
    "Bob",
    company,
    {"theme": "dark", "shortcuts": ["ctrl+k"]},
)

original = {e}
copied = deepcopy(original)

copied_e = next(iter(copied))

assert copied_e is not e
assert copied_e.company is company
assert copied_e.preferences is not e.preferences
assert copied_e.preferences["shortcuts"] is not e.preferences["shortcuts"]


## Problem 11 — Implement selective `__deepcopy__`

Create a class `Session` with:

- `name`;
- `shared_cache`, which should remain shared;
- `state`, which should be deeply copied.

Implement `__deepcopy__` correctly using `memo`, then prove the desired identity relationships.


### Solution 11


In [27]:
class Session:
    def __init__(self, name, shared_cache, state):
        self.name = name
        self.shared_cache = shared_cache
        self.state = state

    def __deepcopy__(self, memo):
        if id(self) in memo:
            return memo[id(self)]

        clone = type(self).__new__(type(self))
        memo[id(self)] = clone

        clone.name = self.name
        clone.shared_cache = self.shared_cache
        clone.state = deepcopy(self.state, memo)

        return clone


cache = {"compiled": {}}
session = Session(
    "S1",
    cache,
    {"history": [{"action": "open"}]},
)

sessions = {session}
sessions_copy = deepcopy(sessions)

copied_session = next(iter(sessions_copy))

assert copied_session is not session
assert copied_session.shared_cache is session.shared_cache
assert copied_session.state is not session.state
assert copied_session.state["history"] is not session.state["history"]
assert copied_session.state["history"][0] is not session.state["history"][0]


## 15. A subtle equality problem

When a class defines value equality, separate copied objects may still compare equal.


In [28]:
@dataclass(frozen=True)
class Token:
    value: str


t1 = Token("abc")
original = {t1}
copied = deepcopy(original)

copied_t1 = next(iter(copied))

print("same identity:", copied_t1 is t1)
print("equal value:", copied_t1 == t1)

assert copied_t1 == t1


same identity: False
equal value: True


This is why `==` and `is` answer different questions:

- `a == b` asks whether values are considered equal.
- `a is b` asks whether both references point to the exact same object.


## Problem 12 — Identity versus equality test design

Write a function that determines whether two sets:

1. are equal by set value semantics;
2. are distinct outer containers;
3. share any element identities.

Use it on a shallow copy and a deep copy.


### Solution 12


In [29]:
def compare_sets(left, right):
    left_ids = ids_of(left)
    right_ids = ids_of(right)

    return {
        "equal": left == right,
        "same_container": left is right,
        "shares_element_identity": bool(left_ids & right_ids),
    }


a = Person("A")
b = Person("B")
original = {a, b}

shallow = original.copy()
deep = deepcopy(original)

print("shallow:", compare_sets(original, shallow))
print("deep:", compare_sets(original, deep))

assert compare_sets(original, shallow) == {
    "equal": True,
    "same_container": False,
    "shares_element_identity": True,
}

# Person uses identity equality, so the deep copy is not value-equal as a set.
assert compare_sets(original, deep)["same_container"] is False
assert compare_sets(original, deep)["shares_element_identity"] is False


shallow: {'equal': True, 'same_container': False, 'shares_element_identity': True}
deep: {'equal': False, 'same_container': False, 'shares_element_identity': False}


### Important observation

Our original `Person` class does not implement value equality, so distinct deep-copied `Person` objects compare unequal by identity. Therefore the original and deep-copied sets are not necessarily `==`.

That is not a failure of deep copying; it is a consequence of the class's equality semantics.


## 16. Value equality changes the result

Let's define a value-like mutable class. Because it defines `__eq__`, we must also deliberately define a compatible hash if we want to use instances in sets.

Here we will keep the hash based on an immutable identifier and allow only a non-key attribute to mutate.


In [30]:
class Account:
    def __init__(self, account_id, display_name):
        self.account_id = account_id
        self.display_name = display_name

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented
        return self.account_id == other.account_id

    def __hash__(self):
        return hash(self.account_id)

    def __repr__(self):
        return (
            f"Account(account_id={self.account_id!r}, "
            f"display_name={self.display_name!r})"
        )


a1 = Account("A-001", "Alice")
accounts = {a1}
accounts_deep = deepcopy(accounts)

assert accounts == accounts_deep
assert ids_of(accounts).isdisjoint(ids_of(accounts_deep))

a1.display_name = "Alice Cooper"

assert accounts == accounts_deep
print(accounts)
print(accounts_deep)


{Account(account_id='A-001', display_name='Alice Cooper')}
{Account(account_id='A-001', display_name='Alice')}


## Problem 13 — Stable-hash mutable object

Design a class `Record` that:

- is stored in sets;
- compares by immutable `record_id`;
- hashes by `record_id`;
- allows a mutable `payload`;
- remains safe if only `payload` changes.

Then deep-copy a set of records and verify independence of payload data.


### Solution 13


In [31]:
class Record:
    def __init__(self, record_id, payload):
        self.record_id = record_id
        self.payload = dict(payload)

    def __eq__(self, other):
        if not isinstance(other, Record):
            return NotImplemented
        return self.record_id == other.record_id

    def __hash__(self):
        return hash(self.record_id)

    def __repr__(self):
        return f"Record({self.record_id!r}, {self.payload!r})"


r1 = Record(1, {"count": 10})
r2 = Record(2, {"count": 20})

records = {r1, r2}
records_copy = deepcopy(records)

assert records == records_copy
assert ids_of(records).isdisjoint(ids_of(records_copy))

r1.payload["count"] = 999

copy_by_id = {record.record_id: record for record in records_copy}

assert copy_by_id[1].payload["count"] == 10
assert r1 in records  # hash still stable

print(records)
print(records_copy)


{Record(1, {'count': 999}), Record(2, {'count': 20})}
{Record(1, {'count': 10}), Record(2, {'count': 20})}


## 17. Copying sets with immutable tuples

Tuples are hashable only when all their elements are hashable.


In [32]:
valid = {
    (1, 2),
    ("a", "b"),
    (True, None),
}

assert (1, 2) in valid

try:
    invalid_tuple = (1, [2, 3])
    {invalid_tuple}
except TypeError as exc:
    print(type(exc).__name__, exc)


TypeError unhashable type: 'list'


## Problem 14 — Find all valid set elements

Which of these can be inserted directly into a set?

```python
1
"abc"
(1, 2)
(1, [2, 3])
frozenset({1, 2})
{1, 2}
["a", "b"]
Person("Ada")
```

Explain each answer.


### Solution 14

Directly valid:

- `1` — integers are hashable.
- `"abc"` — strings are hashable.
- `(1, 2)` — tuple contents are hashable.
- `frozenset({1, 2})` — frozensets are hashable when their elements are hashable.
- `Person("Ada")` — the simple `Person` class uses identity-based hashing by default.

Invalid:

- `(1, [2, 3])` — contains an unhashable list.
- `{1, 2}` — normal sets are unhashable.
- `["a", "b"]` — lists are unhashable.


## 18. Performance: shallow copy is O(n) in the set size

Creating a shallow copy must create a new hash table and insert references to the existing elements. Conceptually, this scales linearly with the number of elements.

A deep copy may be much more expensive because it recursively traverses the reachable object graph.


In [33]:
import timeit

small = set(range(1_000))
large = set(range(100_000))

small_time = timeit.timeit("s.copy()", globals={"s": small}, number=100)
large_time = timeit.timeit("s.copy()", globals={"s": large}, number=10)

print("100 copies of 1,000-element set:", round(small_time, 6), "seconds")
print("10 copies of 100,000-element set:", round(large_time, 6), "seconds")


100 copies of 1,000-element set: 0.000566 seconds
10 copies of 100,000-element set: 0.03496 seconds


Benchmark results vary by machine and Python version. Use benchmarking to compare strategies in your actual workload, not to memorize absolute timing numbers.


## Problem 15 — Benchmark three shallow-copy methods

Benchmark:

- `s.copy()`
- `set(s)`
- `{*s}`

Use the same source set and enough repetitions to reduce noise.

Do not assume one method will always dominate across all Python versions.


### Solution 15


In [34]:
source = set(range(50_000))

benchmarks = {
    "copy()": timeit.timeit("source.copy()", globals=globals(), number=100),
    "set(source)": timeit.timeit("set(source)", globals=globals(), number=100),
    "{*source}": timeit.timeit("{*source}", globals=globals(), number=100),
}

for name, seconds in sorted(benchmarks.items(), key=lambda item: item[1]):
    print(f"{name:12} {seconds:.6f} s")


copy()       0.166087 s
{*source}    0.213226 s
set(source)  0.230329 s


## 19. Defensive copying at API boundaries

A function may copy an incoming set when it needs to protect itself from later **membership changes** made by the caller.


In [35]:
class Registry:
    def __init__(self, members):
        self._members = set(members)

    @property
    def members(self):
        # Return a shallow copy so callers cannot directly mutate
        # the registry's internal set structure.
        return self._members.copy()


a = Person("A")
b = Person("B")

external = {a, b}
registry = Registry(external)

external.clear()

assert len(external) == 0
assert len(registry.members) == 2


However, because the copy is shallow, callers still receive references to the same member objects. If the members themselves must be protected, a different ownership or immutability design is needed.


## Problem 16 — Defensive-copy bug

The following property looks safe:

```python
@property
def users(self):
    return self._users.copy()
```

But callers can still mutate `User` objects stored inside the returned set.

Give three design options when such mutation must be prevented.


### Solution 16

Possible designs include:

1. Store immutable/frozen user value objects.
2. Return immutable projections such as `frozenset` of immutable DTOs/records.
3. Return deep copies when the object graph is suitable and the cost is acceptable.

Often the best design is immutability rather than repeated deep copying.


## 20. `frozenset` for read-only membership snapshots

If clients need a membership snapshot they cannot mutate, return a `frozenset`.


In [36]:
class ImmutableRegistryView:
    def __init__(self, members):
        self._members = set(members)

    @property
    def members(self):
        return frozenset(self._members)


registry = ImmutableRegistryView({1, 2, 3})
view = registry.members

assert isinstance(view, frozenset)

try:
    view.add(4)
except AttributeError as exc:
    print(type(exc).__name__, exc)


AttributeError 'frozenset' object has no attribute 'add'


This protects the outer collection structure, but mutable objects inside a frozenset can still mutate if they are hashable and externally accessible.


## Problem 17 — Copying while filtering

Given a set of immutable integers, create a new independent set containing only even values.

Compare the intent of:

```python
copy = s.copy()
copy = {x for x in s if x % 2 == 0}
```

Which is appropriate for “copy and filter”?


### Solution 17

A set comprehension is appropriate because the requested operation is not merely copying—it is a transformation.


In [37]:
s = set(range(10))

evens = {x for x in s if x % 2 == 0}

assert evens == {0, 2, 4, 6, 8}
assert evens is not s


## 21. Copying while mapping

Likewise, use a comprehension for a semantic transformation.


In [38]:
names = {"alice", "bob", "charlie"}

upper_names = {name.upper() for name in names}

assert upper_names == {"ALICE", "BOB", "CHARLIE"}


## Problem 18 — Deep copy or comprehension?

You have:

```python
prices = {10, 20, 30}
```

You need a new set with VAT added by multiplying every price by `1.2`.

Should you use `deepcopy`?


### Solution 18

No. This is a transformation, not an ownership-copy problem.

Use a set comprehension.


In [39]:
prices = {10, 20, 30}
with_vat = {price * 1.2 for price in prices}

assert with_vat == {12.0, 24.0, 36.0}


## 22. Set algebra already returns new sets in non-mutating forms

Operations such as `|`, `&`, `-`, and `^` return new sets.


In [40]:
a = {1, 2, 3}
b = {3, 4, 5}

union = a | b
intersection = a & b
difference = a - b
symmetric_difference = a ^ b

assert union == {1, 2, 3, 4, 5}
assert intersection == {3}
assert difference == {1, 2}
assert symmetric_difference == {1, 2, 4, 5}

assert union is not a
assert union is not b


By contrast, methods such as `update`, `intersection_update`, and `difference_update` mutate an existing set.


## Problem 19 — Preserve the original set

Starting with:

```python
allowed = {"read", "write"}
extra = {"admin"}
```

Create a result containing all permissions without modifying either input set.

Then demonstrate a mutating version and explain the difference.


### Solution 19


In [41]:
allowed = {"read", "write"}
extra = {"admin"}

combined = allowed | extra

assert allowed == {"read", "write"}
assert extra == {"admin"}
assert combined == {"read", "write", "admin"}

mutating = allowed.copy()
mutating.update(extra)

assert mutating == combined
assert allowed == {"read", "write"}


## 23. Deepcopy and immutable elements

Deep copying a set of immutable values may provide little or no semantic benefit over a shallow copy.


In [42]:
immutable_values = {
    1,
    "python",
    (2, 3),
    frozenset({4, 5}),
}

shallow = immutable_values.copy()
deep = deepcopy(immutable_values)

assert shallow == immutable_values
assert deep == immutable_values
assert shallow is not immutable_values
assert deep is not immutable_values


## Problem 20 — Review a wasteful copy

A code review contains:

```python
permissions = {"read", "write", "execute"}
safe_permissions = deepcopy(permissions)
```

Is this wrong? Is it useful? What would you recommend?


### Solution 20

It is not inherently incorrect, but it is unnecessarily heavy.

All elements are immutable strings, so:

```python
safe_permissions = permissions.copy()
```

is clearer and sufficient if a new mutable outer set is required.

If callers should not modify the collection at all, consider a `frozenset` instead.


## 24. Advanced invariant checker

Let's build a reusable utility that inspects copy relationships.


In [43]:
def copy_relationship(original, candidate):
    original_ids = ids_of(original)
    candidate_ids = ids_of(candidate)

    return {
        "same_outer_object": original is candidate,
        "same_length": len(original) == len(candidate),
        "shared_element_count": len(original_ids & candidate_ids),
        "all_element_identities_shared": original_ids == candidate_ids,
        "no_element_identities_shared": original_ids.isdisjoint(candidate_ids),
    }


items = {Person("A"), Person("B"), Person("C")}

print("assignment:")
print(copy_relationship(items, items))

print("\nshallow:")
print(copy_relationship(items, items.copy()))

print("\ndeep:")
print(copy_relationship(items, deepcopy(items)))


assignment:
{'same_outer_object': True, 'same_length': True, 'shared_element_count': 3, 'all_element_identities_shared': True, 'no_element_identities_shared': False}

shallow:
{'same_outer_object': False, 'same_length': True, 'shared_element_count': 3, 'all_element_identities_shared': True, 'no_element_identities_shared': False}

deep:
{'same_outer_object': False, 'same_length': True, 'shared_element_count': 0, 'all_element_identities_shared': False, 'no_element_identities_shared': True}


## Problem 21 — Write `assert_shallow_copy`

Implement a function `assert_shallow_copy(original, candidate)` that verifies:

- different outer object;
- same number of elements;
- exactly the same element identities.

Do not depend on order.


### Solution 21


In [44]:
def assert_shallow_copy(original, candidate):
    assert candidate is not original
    assert len(candidate) == len(original)
    assert ids_of(candidate) == ids_of(original)


items = {Person("A"), Person("B")}
candidate = items.copy()

assert_shallow_copy(items, candidate)
print("Shallow-copy checks passed.")


Shallow-copy checks passed.


## Problem 22 — Write `assert_deeply_separated`

Implement a helper that verifies that two sets:

- are distinct containers;
- have the same length;
- share no top-level element identities.

Explain why this is still not enough to prove that the entire nested object graphs are fully independent.


### Solution 22


In [45]:
def assert_deeply_separated(original, candidate):
    assert candidate is not original
    assert len(candidate) == len(original)
    assert ids_of(original).isdisjoint(ids_of(candidate))


items = {Person("A"), Person("B")}
candidate = deepcopy(items)

assert_deeply_separated(items, candidate)
print("Top-level deep-separation checks passed.")


Top-level deep-separation checks passed.


This helper checks only the identities of the set's direct elements. Those copied elements might still deliberately or accidentally share nested references. Full graph independence requires checking the ownership relationships that matter for the application.


## 25. Advanced challenge — selective graph ownership

Consider a `Document` with:

- immutable `document_id`;
- mutable `content`;
- shared immutable `Schema`;
- shared mutable `Cache` that should stay shared by design.

A blind deep copy may violate intended ownership by duplicating objects that should remain shared.

### Task

Implement:

1. stable equality/hash based on `document_id`;
2. a `clone()` method;
3. independent content;
4. shared schema;
5. shared cache;
6. tests proving all identity relationships.


### Solution 25


In [46]:
@dataclass(frozen=True)
class Schema:
    name: str
    version: int


class Cache:
    def __init__(self):
        self.data = {}


class Document:
    def __init__(self, document_id, content, schema, cache):
        self.document_id = document_id
        self.content = dict(content)
        self.schema = schema
        self.cache = cache

    def __eq__(self, other):
        if not isinstance(other, Document):
            return NotImplemented
        return self.document_id == other.document_id

    def __hash__(self):
        return hash(self.document_id)

    def clone(self):
        return Document(
            document_id=self.document_id,
            content=deepcopy(self.content),
            schema=self.schema,
            cache=self.cache,
        )

    def __repr__(self):
        return f"Document({self.document_id!r})"


schema = Schema("article", 1)
cache = Cache()

doc = Document(
    "DOC-1",
    {"sections": [{"title": "Intro"}]},
    schema,
    cache,
)

docs = {doc}
cloned_docs = {d.clone() for d in docs}
clone = next(iter(cloned_docs))

assert clone is not doc
assert clone == doc
assert clone.schema is doc.schema
assert clone.cache is doc.cache
assert clone.content is not doc.content
assert clone.content["sections"] is not doc.content["sections"]
assert clone.content["sections"][0] is not doc.content["sections"][0]

doc.content["sections"][0]["title"] = "Changed"

assert clone.content["sections"][0]["title"] == "Intro"

print("Original:", doc.content)
print("Clone:", clone.content)


Original: {'sections': [{'title': 'Changed'}]}
Clone: {'sections': [{'title': 'Intro'}]}


## 26. Advanced challenge — repair a mutable-hash class

The following class is unsafe:

```python
class Product:
    def __init__(self, sku, price):
        self.sku = sku
        self.price = price

    def __eq__(self, other):
        return self.sku == other.sku and self.price == other.price

    def __hash__(self):
        return hash((self.sku, self.price))
```

The business allows `price` to change.

### Task

Redesign equality/hash semantics so instances remain safe in sets while prices change.


### Solution 26

If `sku` is the immutable business identity, equality and hashing should use only `sku`.


In [47]:
class Product:
    def __init__(self, sku, price):
        self.sku = sku
        self.price = price

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.sku == other.sku

    def __hash__(self):
        return hash(self.sku)

    def __repr__(self):
        return f"Product({self.sku!r}, price={self.price!r})"


product = Product("SKU-001", 10.0)
products = {product}

old_hash = hash(product)
product.price = 12.5
new_hash = hash(product)

assert old_hash == new_hash
assert product in products

print(products)


{Product('SKU-001', price=12.5)}


## 27. Advanced challenge — detect accidental aliasing

Suppose a function claims to return a fully independent copy of a set of objects. Write a test that catches a shallow-copy implementation.


### Solution 27


In [48]:
class MutableItem:
    def __init__(self, value):
        self.value = value


def buggy_clone(items):
    return items.copy()


def correct_clone(items):
    return deepcopy(items)


def assert_no_top_level_aliasing(original, clone):
    assert clone is not original
    assert ids_of(original).isdisjoint(ids_of(clone))


items = {MutableItem(1), MutableItem(2)}

try:
    assert_no_top_level_aliasing(items, buggy_clone(items))
except AssertionError:
    print("Correctly detected accidental shallow copying.")

assert_no_top_level_aliasing(items, correct_clone(items))
print("Deep clone passed.")


Correctly detected accidental shallow copying.
Deep clone passed.


## 28. Advanced challenge — copy a set without `copy()`

Implement three functions:

- `copy_with_constructor`
- `copy_with_unpacking`
- `copy_with_comprehension`

Then verify all are shallow copies.


### Solution 28


In [49]:
def copy_with_constructor(s):
    return set(s)


def copy_with_unpacking(s):
    return {*s}


def copy_with_comprehension(s):
    return {item for item in s}


items = {Person("A"), Person("B")}

for copier in [
    copy_with_constructor,
    copy_with_unpacking,
    copy_with_comprehension,
]:
    candidate = copier(items)
    assert_shallow_copy(items, candidate)

print("All implementations are shallow copies.")


All implementations are shallow copies.


### Style note

A comprehension works, but `s.copy()` is usually clearer when the intent is simply “copy this set.” Use a comprehension when filtering or transforming elements.


## 29. Advanced challenge — snapshot versus clone

A system holds a set of mutable `Task` objects.

Define the difference between:

- a **membership snapshot**;
- an **independent object clone**.

Implement both.


### Solution 29


In [50]:
class Task:
    def __init__(self, title, done=False):
        self.title = title
        self.done = done

    def __repr__(self):
        return f"Task({self.title!r}, done={self.done})"


tasks = {Task("Study"), Task("Exercise")}

membership_snapshot = tasks.copy()
independent_clone = deepcopy(tasks)

original_task = next(task for task in tasks if task.title == "Study")
original_task.done = True

snapshot_task = next(task for task in membership_snapshot if task.title == "Study")
clone_task = next(task for task in independent_clone if task.title == "Study")

assert snapshot_task is original_task
assert snapshot_task.done is True

assert clone_task is not original_task
assert clone_task.done is False

print("Original:", tasks)
print("Membership snapshot:", membership_snapshot)
print("Independent clone:", independent_clone)


Original: {Task('Exercise', done=False), Task('Study', done=True)}
Membership snapshot: {Task('Exercise', done=False), Task('Study', done=True)}
Independent clone: {Task('Exercise', done=False), Task('Study', done=False)}


## 30. Advanced challenge — immutable snapshot DTOs

Instead of deep-copying mutable domain objects, expose immutable snapshots.


In [51]:
@dataclass(frozen=True)
class TaskSnapshot:
    title: str
    done: bool


def snapshot_tasks(tasks):
    return frozenset(
        TaskSnapshot(task.title, task.done)
        for task in tasks
    )


tasks = {Task("Study"), Task("Exercise")}
snapshot = snapshot_tasks(tasks)

study = next(task for task in tasks if task.title == "Study")
study.done = True

# The immutable snapshot retains the values from snapshot time.
assert TaskSnapshot("Study", False) in snapshot

print(snapshot)


frozenset({TaskSnapshot(title='Study', done=False), TaskSnapshot(title='Exercise', done=False)})


This is a common production-grade design: expose purpose-built immutable snapshots instead of copying an entire mutable domain graph.


# Capstone Problem

You are implementing a collaborative editor.

Each `EditorUser` has:

- immutable `user_id`;
- mutable `display_name`;
- mutable `preferences`;
- shared `Organization` reference.

The application stores users in a set.

Requirements:

1. Changing `display_name` must not corrupt set membership.
2. Changing `preferences` must not corrupt set membership.
3. A membership snapshot should use the same user objects.
4. A backup clone should create independent user objects and independent preferences.
5. The organization should remain shared even in backup clones.
6. The implementation must not rely on set iteration order.

Implement the model and all tests.


## Capstone Solution


In [52]:
class Organization:
    def __init__(self, org_id, name):
        self.org_id = org_id
        self.name = name

    def __repr__(self):
        return f"Organization({self.org_id!r}, {self.name!r})"


class EditorUser:
    def __init__(self, user_id, display_name, preferences, organization):
        self.user_id = user_id
        self.display_name = display_name
        self.preferences = dict(preferences)
        self.organization = organization

    def __eq__(self, other):
        if not isinstance(other, EditorUser):
            return NotImplemented
        return self.user_id == other.user_id

    def __hash__(self):
        # Hash uses only the immutable identity field.
        return hash(self.user_id)

    def __deepcopy__(self, memo):
        if id(self) in memo:
            return memo[id(self)]

        clone = type(self).__new__(type(self))
        memo[id(self)] = clone

        clone.user_id = self.user_id
        clone.display_name = self.display_name
        clone.preferences = deepcopy(self.preferences, memo)
        clone.organization = self.organization  # intentionally shared

        return clone

    def __repr__(self):
        return (
            f"EditorUser({self.user_id!r}, "
            f"display_name={self.display_name!r})"
        )


org = Organization("ORG-1", "Example Org")

alice = EditorUser(
    "U-1",
    "Alice",
    {"theme": "dark", "shortcuts": ["save"]},
    org,
)

bob = EditorUser(
    "U-2",
    "Bob",
    {"theme": "light", "shortcuts": ["search"]},
    org,
)

users = {alice, bob}

# Requirement 1: display_name mutation is safe.
alice_hash_before = hash(alice)
alice.display_name = "Alice Cooper"
alice_hash_after = hash(alice)

assert alice_hash_before == alice_hash_after
assert alice in users

# Requirement 2: preference mutation is safe for membership.
alice.preferences["theme"] = "system"
assert alice in users

# Requirement 3: shallow membership snapshot shares users.
membership_snapshot = users.copy()

assert membership_snapshot is not users
assert ids_of(membership_snapshot) == ids_of(users)

# Requirement 4 + 5: backup clone deep-copies users/preferences,
# while preserving shared Organization.
backup = deepcopy(users)

assert backup is not users
assert ids_of(backup).isdisjoint(ids_of(users))

original_by_id = {user.user_id: user for user in users}
backup_by_id = {user.user_id: user for user in backup}

for user_id in original_by_id:
    original_user = original_by_id[user_id]
    backup_user = backup_by_id[user_id]

    assert backup_user is not original_user
    assert backup_user.preferences is not original_user.preferences
    assert backup_user.organization is original_user.organization

# Mutating original nested state does not affect backup.
original_by_id["U-1"].preferences["shortcuts"].append("open-command")

assert backup_by_id["U-1"].preferences["shortcuts"] == ["save"]

print("Current users:")
for user in sorted(users, key=lambda user: user.user_id):
    print(" ", user, user.preferences)

print("\nBackup users:")
for user in sorted(backup, key=lambda user: user.user_id):
    print(" ", user, user.preferences)


Current users:
  EditorUser('U-1', display_name='Alice Cooper') {'theme': 'system', 'shortcuts': ['save', 'open-command']}
  EditorUser('U-2', display_name='Bob') {'theme': 'light', 'shortcuts': ['search']}

Backup users:
  EditorUser('U-1', display_name='Alice Cooper') {'theme': 'system', 'shortcuts': ['save']}
  EditorUser('U-2', display_name='Bob') {'theme': 'light', 'shortcuts': ['search']}


# Best-Practice Summary

1. **Assignment is not copying.**
   `b = a` creates a second reference to the same set.

2. **Use `s.copy()` for a straightforward shallow copy.**
   `set(s)` and `{*s}` are also shallow.

3. **A shallow copy duplicates only the outer set.**
   The element objects remain shared.

4. **Use `deepcopy()` only when recursive ownership independence is actually required.**

5. **Prefer explicit cloning for domain models.**
   It makes shared versus independent references intentional.

6. **Never mutate fields that participate in an object's hash while the object is inside a set.**

7. **Prefer immutable value objects for set keys.**
   Frozen dataclasses are often a strong choice.

8. **Do not rely on set iteration/display order.**

9. **Test identity and equality separately.**
   `is` and `==` answer different questions.

10. **Use `frozenset` for immutable membership views.**

11. **Use comprehensions for transformations, not merely for stylistic copying.**

12. **Deep copying has cost and semantic consequences.**
    Copy only as deeply as the ownership model requires.


# Final Self-Check Exercises

Try these without looking back:

1. Why can a mutable instance of a normal user-defined class be stored in a set?
2. Why is mutating a field used by `__hash__` dangerous?
3. What exactly does `set.copy()` duplicate?
4. When is a shallow copy preferable to `deepcopy()`?
5. What problem does `deepcopy`'s memo table solve?
6. Why can two deep-copied sets fail `==` even if they “look the same”?
7. When would `frozenset` be better than returning a copied `set`?
8. Why is explicit domain cloning sometimes safer than generic `deepcopy()`?
9. How would you test that a supposed deep copy does not share top-level elements?
10. Why should set-based tests avoid assuming iteration order?


## Final Self-Check Answers

1. Normal user-defined objects are typically hashable by identity unless equality/hash behavior is overridden.
2. Sets use the hash to locate elements. Changing it after insertion violates the hash-table invariant.
3. The outer set structure only; element references are reused.
4. When elements may safely be shared, especially immutable values or intentionally shared domain entities.
5. It preserves cycles and shared-reference relationships while preventing infinite recursion.
6. Equality behavior is defined by the element class. Identity-based objects may be unequal after copying.
7. When consumers need read-only membership rather than a mutable copy.
8. It allows precise control over which references are copied and which intentionally remain shared.
9. Compare `id` sets and assert they are disjoint.
10. Sets are unordered; order is not part of their semantic contract.
